# **Multi-agent Financial Advisor: Full Pipeline Demo**

**Fordham MSQF Capstone Summer 2026**

Six AI agents working in sequence to build a personalized portfolio recommendation:

```
Research Agent  →  Profile Agent  →  Allocation Agent  →  Risk Agent  →  Compliance Agent  →  AdvisorPackage
```

**Every number is deterministic**, LLMs writes rationale text and reasoning traces only.

---
### Changes:
- **Regime-adaptive risk caps**: 60-day rolling vol ratio classifies market as NORMAL / ELEVATED / CRISIS.  
  Drawdown caps widen 25 % (ELEVATED) or 50 % (CRISIS) to prevent procyclical selling at market bottoms.
- **Risk-profile downgrade loop**: CRITICAL stress breach steps AGGRESSIVE → MODERATE → CONSERVATIVE  
  one notch per FLAG iteration; reverts automatically on a fresh run.

---
### Prerequisites
1. `pip install -r requirements.txt`
2. **First run only**: To populate the data cache, run Section 1 below (~10 min, prompts for WRDS login credentials)
3. Optional env vars:
   - `FRED_API_KEY`: free at fred.stlouisfed.org; falls back to 4.4 % DGS10 if missing
   - `ANTHROPIC_API_KEY`: Claude API; LLM rationale cells are skipped gracefully if missing

### **Environment Setup**

In [1]:
import os, sys, importlib, logging
from pathlib import Path

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load .env file if present — pip install python-dotenv
try:
    from dotenv import load_dotenv
    load_dotenv(Path(PROJECT_ROOT) / '.env')
    print('.env loaded')
except ImportError:
    print('dotenv not installed — run: pip install python-dotenv')

importlib.invalidate_caches()
logging.basicConfig(level=logging.WARNING)

FRED_KEY      = os.environ.get('FRED_API_KEY')
ANTHROPIC_KEY = os.environ.get('ANTHROPIC_API_KEY')
print(f'FRED_API_KEY set:      {bool(FRED_KEY)}')
print(f'ANTHROPIC_API_KEY set: {bool(ANTHROPIC_KEY)}')

.env loaded
FRED_API_KEY set:      True
ANTHROPIC_API_KEY set: True


In [2]:
# Check if cache is populated
REQUIRED = [
    'crsp_monthly.parquet', 'crsp_daily.parquet', 'ff_risk_factors.parquet',
    'fred_macro.parquet',   'bls_oes.parquet',    'ff12_monthly.parquet',
    'permno_map.json',      'mkt_cap_weights.json',
]
storage = Path('data/storage')

missing = [f for f in REQUIRED if not (storage / f).exists()]
if not missing:
    print('All data files present, skip to Section 2.')
else:
    for f in REQUIRED:
        mark = 'OK     ' if (storage / f).exists() else 'MISSING'
        print(f'  {mark}  {f}')

All data files present, skip to Section 2.


## **Section 01: One-Time Data Cache Setup**

The pipeline reads from `data/storage/` (gitignored, ~500 MB).  
Run once to populate, then skip this section.

| File | Source | Used by |
|---|---|---|
| `crsp_monthly.parquet` | WRDS/CRSP | Allocation — BL market-cap prior weights |
| `crsp_daily.parquet` | WRDS/CRSP | Risk — VaR, drawdown, regime detection |
| `ff_risk_factors.parquet` | WRDS/FF | Allocation + Risk — 4-factor model |
| `ff12_monthly.parquet` | Ken French | Research — regime feature validation |
| `bls_oes.parquet` | BLS OES 2023 | Profile — salary distributions |
| `fred_macro.parquet` | FRED | Research — 13 macro series |
| `permno_map.json` | WRDS | Risk — ticker → CRSP permno |
| `mkt_cap_weights.json` | WRDS | Allocation — BL equilibrium prior |

In [ ]:
# In case cache isn't populated, section 1 runs
# WRDS connection(prompts for username + password on first run)
from data.fetch.wrds import get_connection
conn = get_connection()                     # Older syntax, need to change this
print('Connected to WRDS')

In [ ]:
from agents.allocation.adapters import DEFAULT_TICKERS
from data.fetch.wrds import fetch_crsp_monthly, fetch_crsp_daily, fetch_ff_factors

fetch_crsp_monthly(DEFAULT_TICKERS, conn=conn)   # crsp_monthly.parquet + permno_map + mkt_cap_weights
fetch_crsp_daily(DEFAULT_TICKERS, conn=conn)     # crsp_daily.parquet  (~5 min)
fetch_ff_factors(conn=conn)                      # ff_risk_factors.parquet(data set that breaks 
                                                 # the whole US market into 12 industry portfolios)
print('WRDS fetch done')

In [ ]:
from data.fetch.fred import fetch_fred_macro
from data.fetch.bls  import fetch_bls_oes
from data.fetch.factors import fetch_ff12

# Fetch FRED and BLS data
fetch_fred_macro(fred_api_key=FRED_KEY)
fetch_bls_oes()
fetch_ff12()
print('Public data fetch done')

## **Section 02: Full Pipeline Run**

---
### **Agent 1: Research Agent(Macro Regime Detection)**

**What it does:**  
Pulls 13 FRED macro series (yield curve, unemployment, CPI, credit spread), runs PELT change-point detection to find structural breaks, clusters segments with K-means, then trains an XGBoost classifier against five historically-labelled anchor windows:

| Label | Anchor window | Macro signature |
|---|---|---|
| Early Recovery | 2003-06 – 2004-06 | Low rates, spreads narrowing post dot-com |
| Late-Cycle Expansion | 2005-01 – 2007-06 | Rising rates, low VIX, pre-GFC boom |
| Financial Crisis & ZLB | 2008-09 – 2010-12 | Acute GFC + zero interest rate policy |
| Moderate Expansion | 2015-01 – 2019-06 | Post-QE normalisation, stable growth |
| Inflation Shock | 2022-01 – 2023-06 | Peak CPI, aggressive Fed tightening |

A 6-month rolling majority vote smooths month-to-month noise at regime boundaries.

**Output:** `MacroRegimeSnapshot` passed unchanged to every downstream agent.

In [3]:
from agents.research.research_agent import run_research_agent

# compare_models=False skips HMM/GMM benchmarking (paper validation only)
macro = run_research_agent(fred_api_key=FRED_KEY, compare_models=False)

print(f'Regime:          {macro.regime_label}')
print(f'Confidence:      {macro.regime_confidence:.0%}')
print(f'Prior regime:    {macro.prior_regime}')
print(f'Regime change:   {macro.regime_change_detected}')
print(f'Low confidence:  {macro.is_low_confidence}')
print(f'As of:           {macro.as_of}')
print()
print('--- Key FRED signals ---')
print(f'Yield curve (10Y-2Y):  {macro.yield_curve:+.2f} pp')
print(f'Fed funds rate:         {macro.fed_funds:.2f}%')
print(f'Unemployment:           {macro.unemployment:.1f}%')
print(f'CPI YoY:                {macro.cpi:.1f}%')
print(f'Credit spread:          {macro.credit_spread:.2f} pp')

Detected 4 structural breaks:
  2004-09
  2008-01
  2014-09
  2020-12
Optimal K (max silhouette): 2

=== Feature Importance ===
term_spread_z       0.236876
cpi_z               0.226380
credit_spread_z     0.165667
t10yie_z            0.143091
yield_curve_z       0.076510
vix_z               0.069070
gdp_z               0.034293
unemployment_chg    0.014280
t5yie_z             0.012338
indpro_z            0.011542
fed_funds_chg       0.005721
dgs30_chg           0.002515
dgs5_chg            0.001716
Validation complete — Passed: 274 | Failed: 0

=== MacroRegimeSnapshot (most recent month) ===
{
  "as_of": "2025-12-01",
  "regime_label": "Late-Cycle Expansion",
  "prior_regime": "Late-Cycle Expansion",
  "regime_shift_date": "2023-08-01",
  "regime_confidence": 0.902,
  "regime_volatility": 0.0549,
  "yield_curve": 0.71,
  "term_spread": 0.51,
  "fed_funds": 3.72,
  "unemployment": 4.4,
  "cpi": 2.6533,
  "credit_spread": 1.72,
  "is_low_confidence": false,
  "regime_change_detected": f

---
## **Agent 2: Profile Agent(Human Capital Valuation)**

**What it does:**  
Maps BLS OES May 2023 salary data to 9 occupational personas, looks up income-equity beta (β) and correlation (ρ) from a calibrated table (Ibbotson et al. 2007), then computes:

| Formula | Meaning |
|---|---|
| `HC = Salary × [1 − (1+r)^{−n}] / r` | PV of future earnings (annuity, DGS10 discount) |
| `implicit_equity_exposure = hc_share × β` | Equity risk already carried through the career |
| `effective_risk_budget = (FC + HC×(1−σ)) / total_wealth` | Total risk capacity across the balance sheet |
| `portfolio_equity_target = risk_budget − implicit_equity_exposure` | Equity headroom for the investment portfolio |

A tech exec with β = 1.2 already has ~90% implicit equity exposure through career + RSUs, their portfolio should be mostly bonds. A tenured professor with β = 0.05 can hold a much higher equity allocation.

**Output:** `ProfileAgentOutput`

In [4]:
from agents.profile.profile_agent import run_profile_agent
from agents.profile.profile_model import TARGET_OCCUPATIONS
import pandas as pd

profiles = run_profile_agent(fred_api_key=FRED_KEY)
print(f'Built {len(profiles)} profiles')
print('\n')

OCC_LABEL = {f"bls_{occ['soc']}_p50": occ['label'] for occ in TARGET_OCCUPATIONS}

summary = pd.DataFrame([{
    'Occupation':      OCC_LABEL.get(p.client_id, p.client_id),
    'Client':          p.client_id,
    'HC type':         p.human_capital_type.value,
    'β':               f'{p.income_equity_beta:.2f}',
    'Implicit eq exp': f'{p.implicit_equity_exposure:.1%}',
    'Risk budget':     f'{p.effective_risk_budget:.1%}',
    'HC % wealth':     f'{p.human_capital_pct_of_total:.0f}%',
} for p in profiles])
print(summary.to_string(index=False))

Discount rate (FRED DGS10): 0.0418
Loaded BLS OES from parquet cache: C:\Users\nihar\summer_2026\agentic-portfolio-construction-dev\data\storage\bls_oes.parquet
Built 9 personas from 9 SOC codes
Validated 9 / 9 profiles successfully
Saved 9 profiles → C:\Users\nihar\summer_2026\agentic-portfolio-construction-dev\data\outputs\profiles_all.json
Saved 9 profiles → C:\Users\nihar\summer_2026\agentic-portfolio-construction-dev\data\storage\profiles_all.parquet
Built 9 profiles


         Occupation          Client     HC type    β Implicit eq exp Risk budget HC % wealth
  Biology Professor bls_25-1042_p50   bond-like 0.05            4.2%       95.8%         85%
   Registered Nurse bls_29-1141_p50   bond-like 0.05            4.7%       95.3%         94%
 Compliance Officer bls_13-1041_p50   bond-like 0.05            4.6%       95.4%         93%
             Lawyer bls_23-1011_p50       mixed 0.35           31.9%       81.8%         91%
Mechanical Engineer bls_17-2141_p50       mixed 0.35    

In [5]:
# Pick one persona to run through the rest of the pipeline
# SOC 15-1252 = Software Developers (equity-like HC, β ≈ 1.2)
profile = next(p for p in profiles if p.client_id == 'bls_25-1042_p50')

print(f'Selected:               {profile.client_id}')
print(f'Occupation:             {OCC_LABEL.get(profile.client_id, profile.client_id)}')
print(f'Career type:            {profile.career_type}')
print(f'Age / horizon:          {profile.age} yrs / {profile.investment_horizon_years} yrs')
print(f'Financial capital:      ${profile.financial_capital:>12,.0f}')
print(f'Human capital (PV):     ${profile.human_capital_valuation:>12,.0f}')
print(f'Total wealth:           ${profile.total_wealth:>12,.0f}')
print(f'HC type:                {profile.human_capital_type.value}')
print(f'Income beta (β):        {profile.income_equity_beta:.2f}')
print(f'Income-equity corr (ρ): {profile.income_equity_correlation:.2f}')
print(f'Income volatility (σ):  {profile.income_volatility_sigma:.2f}')
print(f'Implicit equity exp:    {profile.implicit_equity_exposure:.1%}')
print(f'Effective risk budget:  {profile.effective_risk_budget:.1%}')
print(f'Risk tolerance:         {profile.risk_tolerance_level.value}')
print(f'Employer sector:        {profile.industry_exposure_sector}')


Selected:               bls_25-1042_p50
Occupation:             Biology Professor
Career type:            Academia
Age / horizon:          47 yrs / 18 yrs
Financial capital:      $     200,000
Human capital (PV):     $   1,095,155
Total wealth:           $   1,295,155
HC type:                bond-like
Income beta (β):        0.05
Income-equity corr (ρ): 0.10
Income volatility (σ):  0.05
Implicit equity exp:    4.2%
Effective risk budget:  95.8%
Risk tolerance:         moderate
Employer sector:        Education


---
### **Agents 3 + 4: Allocation & Risk Loop**

1. **Allocation Agent:** Uses Black-Litterman starting from CRSP market-cap equilibrium weights, runs a 4-factor(MKT/SMB/HML/UMD) model to estimate expected returns, then subtracts `implicit_equity_exposure` so total (career + portfolio) equity risk stays appropriate for the client.

2. **Risk Agent:** Runs deterministic stress tests and returns `APPROVE`, `FLAG`, or `REJECT`

3. **Regime-Adaptive Risk Caps:** Before computing drawdown thresholds, the Risk Agent classifies the current market regime from 60-day rolling portfolio return volatility vs the full-sample baseline:

| Regime | Vol ratio trigger | Drawdown cap multiplier | Example period |
|---|---|---|---|
| NORMAL | < 1.5× | 1.00× (standard) | 2012–2019 |
| ELEVATED | 1.5× - 2.0× | 1.25× (25% wider) | H2 2007, 2011 |
| CRISIS | ≥ 2.0× | 1.50× (50% wider) | Oct 2008, Mar 2020 |

Widening the cap during a crisis prevents the system from forcing a sell at market bottoms, the cap resets automatically once volatility normalises.


**FLAG loop (max 3 iterations):**  
On FLAG, violated constraints are fed back to the Allocation Agent which re-optimises with tighter limits. A CRITICAL stress breach steps the risk profile down one notch (AGGRESSIVE → MODERATE → CONSERVATIVE); the downgrade reverts on a fresh run.

In [6]:
import logging
logging.getLogger().setLevel(logging.INFO)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)

from agents.orchestrator.orchestrator_agent import run_pipeline

pkg = run_pipeline(profile, macro, fred_api_key=FRED_KEY)

logging.getLogger().setLevel(logging.WARNING)
print('\nPipeline complete.')

10:34:15  [pipeline] Iteration 1  (First run)
10:34:30  HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
10:34:30  [pipeline] Allocation done — risky 100.0% E[r] 8.09% vol 15.55%
10:34:42  HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
10:34:42  [pipeline] Risk decision: REJECT
10:34:42  COMPLIANCE [bls_25-1042_p50] — starting checks
10:34:42  COMPLIANCE [bls_25-1042_p50] — Job 1 complete: 0 violation(s), 9 passed
10:34:42  COMPLIANCE [bls_25-1042_p50] — Job 2 complete: 0 total violation(s), 15 passed checks
10:34:42  COMPLIANCE [bls_25-1042_p50] — status: PASS | clearance: True | severity: NONE



Pipeline complete.


---
## **Results**

In [7]:
# Pipeline metadata
m = pkg.metadata
print('=== PIPELINE METADATA ===')
print(f'  Final risk decision:      {m.final_risk_decision.value}')
print(f'  Final compliance status:  {m.final_compliance_status.value}')
print(f'  Risk FLAG revisions:      {m.risk_revisions}')
print(f'  Compliance revisions:     {m.compliance_revisions}')
if m.pipeline_warnings:
    print('  Warnings:')
    for w in m.pipeline_warnings:
        print(f'    - {w}')

=== PIPELINE METADATA ===
  Final risk decision:      REJECT
  Final compliance status:  PASS
  Risk FLAG revisions:      0
  Compliance revisions:     0
  Warnings:
    - Risk Agent REJECTED portfolio after 0 FLAG revision(s)


In [8]:
# Portfolio weights
weights = pkg.allocation.proposed_portfolio
df_w = pd.DataFrame(
    [{'Ticker': t, 'Weight': w, 'Weight %': f'{w:.1%}'}
     for t, w in sorted(weights.items(), key=lambda x: x[1], reverse=True)
     if w > 0.001]
)
print('=== PORTFOLIO WEIGHTS ===')
print(df_w.to_string(index=False))
print(f'\nTotal: {sum(weights.values()):.4f}')

=== PORTFOLIO WEIGHTS ===
Ticker   Weight Weight %
   SPY 0.100000    10.0%
   XLK 0.100000    10.0%
   XLV 0.100000    10.0%
   XLY 0.100000    10.0%
   XLC 0.099134     9.9%
   XLF 0.066955     6.7%
   EFA 0.065935     6.6%
   XLI 0.064145     6.4%
   TLT 0.044892     4.5%
   GLD 0.043282     4.3%
   XLP 0.040800     4.1%
   LQD 0.039791     4.0%
   IWM 0.034065     3.4%
   XLE 0.029576     3.0%
   XLU 0.024247     2.4%
   VNQ 0.024200     2.4%
   TIP 0.017099     1.7%
  XLRE 0.005879     0.6%

Total: 1.0000


In [9]:
# Risk Agent output — includes new regime-adaptive fields
r = pkg.risk
print('=== RISK AGENT OUTPUT ===')
print(f'Decision:    {r.risk_decision.value}')

# ── Regime-adaptive cap (new) ──────────────────────────────────────────
if r.market_regime is not None:
    MULTIPLIERS = {'normal': 1.00, 'elevated': 1.25, 'crisis': 1.50}
    mult = MULTIPLIERS.get(r.market_regime.value, 1.0)
    cap_note = {
        'normal':   'standard cap',
        'elevated': 'widened 25% — elevated vol',
        'crisis':   'widened 50% — crisis vol',
    }.get(r.market_regime.value, '')
    print(f'Market regime:         {r.market_regime.value.upper()}  ({cap_note})')
    print(f'Effective drawdown cap: {r.effective_drawdown_cap:.2%}')

if r.portfolio_volatility_annual is not None:
    print(f'Volatility (annual):    {r.portfolio_volatility_annual:.2%}')

print(f'Violations: {r.violations if r.violations else "None"}')

# ── Regime stress tests (Research Agent taxonomy) ──────────────────────
print('\nRegime stress tests (portfolio loss vs benchmark):')
for regime, ev in r.regime_evaluation.items():
    status = 'PASS' if ev.passed else 'FAIL'
    print(f'  [{status}] {regime:<32}  portfolio {ev.portfolio_drawdown:.1%}  bench {ev.benchmark_drawdown:.1%}')

=== RISK AGENT OUTPUT ===
Decision:    REJECT
Market regime:         NORMAL  (standard cap)
Effective drawdown cap: 20.00%
Volatility (annual):    15.55%
Violations: None

Regime stress tests (portfolio loss vs benchmark):
  [PASS] Early Recovery                    portfolio 29.8%  bench 34.0%
  [PASS] Late-Cycle Expansion              portfolio 10.8%  bench 12.0%
  [PASS] Financial Crisis & ZLB            portfolio 31.7%  bench 54.0%
  [PASS] Moderate Expansion                portfolio 5.4%  bench 6.0%
  [FAIL] Inflation Shock                   portfolio 30.4%  bench 18.0%


In [10]:
# Compliance Agent output
c = pkg.compliance
print('=== COMPLIANCE AGENT OUTPUT ===')
print(f'Clearance:        {c.clearance}')
print(f'Status:           {c.compliance_status.value}')
print(f'Overall severity: {c.overall_severity.value}')
print(f'Recommendation:   {c.recommendation}')

if c.passed_checks:
    sample = ', '.join(c.passed_checks[:5])
    suffix = '...' if len(c.passed_checks) > 5 else ''
    print(f'\nPassed checks ({len(c.passed_checks)}): {sample}{suffix}')

if c.violations:
    print(f'\nViolations ({len(c.violations)}):')
    for v in c.violations:
        print(f'  [{v.severity.value}] {v.check}: {v.description}')

=== COMPLIANCE AGENT OUTPUT ===
Clearance:        True
Status:           PASS
Overall severity: NONE
Recommendation:   Portfolio cleared. All compliance checks passed.

Passed checks (15): check_1_1a_regime_completeness, check_1_1b_position_limit_completeness, check_1_1c_derivation_completeness, check_1_2a_position_limit_consistency, check_1_2b_sector_limit_consistency...


In [11]:
# LLM-generated allocation rationale (sample ticker)
rationale = pkg.allocation.allocation_rationale
if rationale:
    sample_ticker = next(iter(rationale))
    print(f'=== ALLOCATION RATIONALE — {sample_ticker} ===')
    print(rationale[sample_ticker])
else:
    print('No rationale (ANTHROPIC_API_KEY not set)')

=== ALLOCATION RATIONALE — SPY ===
## Portfolio Rationale

With **$1,095,155 in human capital present value** dwarfing the $200,000 financial portfolio and an **income beta of just 0.05** — indicating that the client's educator salary is nearly uncorrelated with equity market movements — the total-wealth framework supports allocating **100% of financial assets to the risky sleeve**, as the bond-like stability of public-sector employment already provides substantial implicit "safe asset" exposure at the total-wealth level. The largest active tilts away from market-cap equilibrium weights are concentrated in **XLC** (+3.9% overweight vs. a 6.0% equilibrium) and **XLY** (+3.5% overweight vs. a 6.5% equilibrium), reflecting deliberate factor views favoring consumer discretionary and communication services sectors, likely driven by expectations of above-equilibrium excess returns in cyclical growth-oriented segments of the market. The portfolio carries an annualized volatility of **15.55%**

---
## **Full Report**

In [12]:
from datetime import date

pro   = pkg.profile
mac   = pkg.macro
alloc = pkg.allocation
risk  = pkg.risk
comp  = pkg.compliance
meta  = pkg.metadata
occ   = OCC_LABEL.get(pro.client_id, pro.client_id)
pet   = pro.portfolio_equity_target if pro.portfolio_equity_target is not None \
        else round(pro.effective_risk_budget - pro.implicit_equity_exposure, 3)

W = 64

def section(title):
    print(f"\n{'=' * W}")
    print(f"  {title}")
    print('=' * W)

def row(label, value, width=36):
    print(f"  {label:<{width}}{value}")

# ── Header ───────────────────────────────────────────────────────────────────
print('=' * W)
print(f"  ADVISOR REPORT — {occ.upper()}")
print(f"  Client: {pro.client_id}   |   Date: {date.today().isoformat()}")
print('=' * W)

# ── 1. Macro Regime ──────────────────────────────────────────────────────────
section("1. MACRO REGIME")
row("Regime:",          mac.regime_label)
row("Confidence:",      f"{mac.regime_confidence:.0%}")
row("As of:",           str(mac.as_of))
if mac.regime_change_detected:
    row("Regime change from:", f"{mac.prior_regime} on {mac.regime_shift_date}")
if mac.is_low_confidence:
    print("  *** WARNING: low confidence regime ***")
print()
row("Yield curve (10Y-2Y):",   f"{mac.yield_curve:+.2f} pp")
row("Term spread (10Y-3M):",   f"{mac.term_spread:+.2f} pp")
row("Fed funds rate:",         f"{mac.fed_funds:.2f}%")
row("Unemployment:",           f"{mac.unemployment:.1f}%")
row("CPI YoY:",                f"{mac.cpi:.1f}%")
row("Credit spread (Baa-10Y):",f"{mac.credit_spread:.2f} pp")

# ── 2. Client Profile ─────────────────────────────────────────────────────────
section("2. CLIENT PROFILE")
row("Occupation:",        occ)
row("Career type:",       pro.career_type)
row("Age:",               f"{pro.age} yrs")
row("Investment horizon:", f"{pro.investment_horizon_years} yrs")
row("Risk tolerance:",    pro.risk_tolerance_level.value)
row("Objective:",         pro.investment_objective.value)
row("Liquidity needs:",   pro.liquidity_needs.value)
row("Pension:",           "Yes" if pro.has_pension else "No")
print()
print(f"  {'-- Wealth Decomposition --':^{W-2}}")
row("Financial capital:",          f"${pro.financial_capital:>14,.0f}   ({100 - pro.human_capital_pct_of_total:.1f}%)")
row("Human capital (PV, DGS10):",  f"${pro.human_capital_valuation:>14,.0f}   ({pro.human_capital_pct_of_total:.1f}%)")
row("Total wealth:",               f"${pro.total_wealth:>14,.0f}   (100%)")
print()
print(f"  {'-- Human Capital Risk Decomposition --':^{W-2}}")
row("HC type:",                    pro.human_capital_type.value)
row("Income volatility (sigma):",  f"{pro.income_volatility_sigma:.2f}")
row("Income-equity beta (beta):",  f"{pro.income_equity_beta:.2f}")
row("Income-equity corr (rho):",   f"{pro.income_equity_correlation:.2f}")
row("Implicit equity exposure:",   f"{pro.implicit_equity_exposure:.1%}")
row("Effective risk budget:",      f"{pro.effective_risk_budget:.1%}   = (FC + HC*(1-sigma)) / TW")
row("Portfolio equity target:",    f"{pet:.1%}   = risk budget - implicit exposure")

# ── 3. Portfolio Allocation ───────────────────────────────────────────────────
section("3. PORTFOLIO ALLOCATION")
weights = alloc.proposed_portfolio
top = sorted(weights.items(), key=lambda x: x[1], reverse=True)
print(f"  {'Ticker':<10} {'Weight':>8}")
print(f"  {'-'*10} {'-'*8}")
for ticker, w in top:
    if w > 0.001:
        print(f"  {ticker:<10} {w:>8.1%}")
print(f"  {'-'*10} {'-'*8}")
print(f"  {'Total':<10} {sum(weights.values()):>8.4f}")

rationale = alloc.allocation_rationale
if rationale:
    print()
    print(f"  -- Allocation Rationale --")
    print()
    text = next(iter(rationale.values()))
    # wrap at ~60 chars
    import textwrap
    for line in textwrap.wrap(text, width=W - 4):
        print(f"  {line}")

# ── 4. Risk Assessment ────────────────────────────────────────────────────────
section("4. RISK ASSESSMENT")
row("Decision:", risk.risk_decision.value)
if risk.market_regime:
    cap_notes = {'normal': 'standard cap', 'elevated': 'widened 25%', 'crisis': 'widened 50%'}
    row("Market regime:", f"{risk.market_regime.value.upper()}  ({cap_notes.get(risk.market_regime.value, '')})")
    row("Effective drawdown cap:", f"{risk.effective_drawdown_cap:.2%}")
if risk.portfolio_volatility_annual is not None:
    row("Portfolio volatility:", f"{risk.portfolio_volatility_annual:.2%}  (annualised)")
print()
print(f"  {'Regime':<32} {'Portfolio':>10} {'Benchmark':>10}  Result")
print(f"  {'-'*32} {'-'*10} {'-'*10}  {'------'}")
for regime_name, ev in risk.regime_evaluation.items():
    result = "PASS" if ev.passed else "FAIL"
    print(f"  {regime_name:<32} {ev.portfolio_drawdown:>10.1%} {ev.benchmark_drawdown:>10.1%}  {result}")
if risk.violations:
    print()
    print("  Violations:")
    for v in risk.violations:
        print(f"    - {v}")

# ── 5. Compliance ─────────────────────────────────────────────────────────────
section("5. COMPLIANCE")
row("Status:",    comp.compliance_status.value)
row("Clearance:", "CLEARED" if comp.clearance else "NOT CLEARED")
row("Severity:",  comp.overall_severity.value)
row("Passed checks:", f"{len(comp.passed_checks)} / {len(comp.passed_checks) + len(comp.violations)}")
print()
import textwrap
for line in textwrap.wrap(comp.recommendation, width=W - 4):
    print(f"  {line}")
if comp.violations:
    print()
    print("  Violations:")
    for v in comp.violations:
        print(f"    [{v.severity.value}] {v.check}")
        for line in textwrap.wrap(v.description, width=W - 8):
            print(f"        {line}")

# ── 6. Pipeline Summary ───────────────────────────────────────────────────────
section("6. PIPELINE SUMMARY")
row("Final risk decision:",     meta.final_risk_decision.value)
row("Final compliance status:", meta.final_compliance_status.value)
row("Risk FLAG revisions:",     str(meta.risk_revisions))
row("Compliance revisions:",    str(meta.compliance_revisions))
if meta.pipeline_warnings:
    print()
    print("  Warnings:")
    for w in meta.pipeline_warnings:
        print(f"    - {w}")

print(f"\n{'=' * W}\n")


  ADVISOR REPORT — BIOLOGY PROFESSOR
  Client: bls_25-1042_p50   |   Date: 2026-07-02

  1. MACRO REGIME
  Regime:                             Late-Cycle Expansion
  Confidence:                         90%
  As of:                              2025-12-01

  Yield curve (10Y-2Y):               +0.71 pp
  Term spread (10Y-3M):               +0.51 pp
  Fed funds rate:                     3.72%
  Unemployment:                       4.4%
  CPI YoY:                            2.7%
  Credit spread (Baa-10Y):            1.72 pp

  2. CLIENT PROFILE
  Occupation:                         Biology Professor
  Career type:                        Academia
  Age:                                47 yrs
  Investment horizon:                 18 yrs
  Risk tolerance:                     moderate
  Objective:                          growth
  Liquidity needs:                    low
  Pension:                            Yes

                    -- Wealth Decomposition --                  
  Financial capita

---
### **Supplementary: Regime-Adaptive Cap Walkthrough**

This section shows how the cap changes across regimes for the selected client, independent of a full pipeline run. Useful for validating the logic and for the thesis writeup.

In [13]:
from contracts import MarketRegime, RiskProfile
from agents.shared.core.risk import MAX_DRAWDOWN_CAP, effective_cap

base_profile = profile.risk_tolerance_level  # e.g. RiskProfile.AGGRESSIVE

print(f'Client risk profile: {base_profile.value}')
print(f'Base drawdown cap:   {MAX_DRAWDOWN_CAP[base_profile]:.0%}')
print()
print(f'{"Regime":<12}  {"Multiplier":<12}  {"Effective cap":<14}  Example period')
print('-' * 65)

examples = {
    MarketRegime.NORMAL:   '2012 – 2019 bull market',
    MarketRegime.ELEVATED: 'H2 2007, Aug 2011 downgrade',
    MarketRegime.CRISIS:   'Oct 2008, Mar 2020 COVID',
}
multipliers = {MarketRegime.NORMAL: 1.00, MarketRegime.ELEVATED: 1.25, MarketRegime.CRISIS: 1.50}

for regime in MarketRegime:
    cap  = effective_cap(base_profile, regime)
    mult = multipliers[regime]
    print(f'{regime.value:<12}  {mult:<12.2f}  {cap:<14.2%}  {examples[regime]}')

Client risk profile: moderate
Base drawdown cap:   20%

Regime        Multiplier    Effective cap   Example period
-----------------------------------------------------------------
normal        1.00          20.00%          2012 – 2019 bull market
elevated      1.25          25.00%          H2 2007, Aug 2011 downgrade
crisis        1.50          30.00%          Oct 2008, Mar 2020 COVID


In [14]:
# Show caps across all three risk profiles × all three regimes
print(f'{"":<12}', end='')
for regime in MarketRegime:
    print(f'  {regime.value.upper():<14}', end='')
print()
print('-' * 58)

for rp in RiskProfile:
    print(f'{rp.value:<12}', end='')
    for regime in MarketRegime:
        cap = effective_cap(rp, regime)
        print(f'  {cap:<14.2%}', end='')
    print()

              NORMAL          ELEVATED        CRISIS        
----------------------------------------------------------
conservative  15.00%          18.75%          22.50%        
moderate      20.00%          25.00%          30.00%        
aggressive    25.00%          31.25%          37.50%        


---
## **Section 3: Run full pipeline for all 9 BLS Personas**

Runs `run_pipeline()` once per profile using the already-computed `macro` snapshot. The Research Agent runs once; the Profile → Allocation → Risk → Compliance chain runs 9 times.

In [15]:
"""from agents.orchestrator.orchestrator_agent import run_pipeline

# macro and profiles are already computed above — no re-run of Research/Profile agents
packages = [run_pipeline(p, macro, fred_api_key=FRED_KEY) for p in profiles]

print(f'{len(packages)} AdvisorPackages produced')
print()

rows = []
for p in packages:
    r = p.risk
    regime_str = r.market_regime.value.upper() if r.market_regime else 'N/A'
    rows.append({
        'Occupation':   OCC_LABEL.get(p.profile.client_id, p.profile.client_id),
        'HC type':      p.profile.human_capital_type.value,
        'Risk':         p.metadata.final_risk_decision.value,
        'Compliance':   p.metadata.final_compliance_status.value,
        'Regime':       regime_str,
        'DD cap':       f'{r.effective_drawdown_cap:.0%}' if r.effective_drawdown_cap else 'N/A',
        'FLAG revs':    p.metadata.risk_revisions,
        'Top holding':  max(p.allocation.proposed_portfolio,
                            key=p.allocation.proposed_portfolio.get),
    })

print(pd.DataFrame(rows).to_string(index=False))"""

"from agents.orchestrator.orchestrator_agent import run_pipeline\n\n# macro and profiles are already computed above — no re-run of Research/Profile agents\npackages = [run_pipeline(p, macro, fred_api_key=FRED_KEY) for p in profiles]\n\nprint(f'{len(packages)} AdvisorPackages produced')\nprint()\n\nrows = []\nfor p in packages:\n    r = p.risk\n    regime_str = r.market_regime.value.upper() if r.market_regime else 'N/A'\n    rows.append({\n        'Occupation':   OCC_LABEL.get(p.profile.client_id, p.profile.client_id),\n        'HC type':      p.profile.human_capital_type.value,\n        'Risk':         p.metadata.final_risk_decision.value,\n        'Compliance':   p.metadata.final_compliance_status.value,\n        'Regime':       regime_str,\n        'DD cap':       f'{r.effective_drawdown_cap:.0%}' if r.effective_drawdown_cap else 'N/A',\n        'FLAG revs':    p.metadata.risk_revisions,\n        'Top holding':  max(p.allocation.proposed_portfolio,\n                            key=p.